In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

In [0]:
yest_date = datetime.now() - timedelta(days=1)
yest_date = yest_date.strftime("%Y%m%d")
sources = ["online", "offline"]
files = ["customers", "orders", "products"]
dbutils.widgets.text("file_run", yest_date, "File Run Parameter")
file_run = dbutils.widgets.get("file_run")
file_run

In [0]:
def load_bronze_data():
    for source in sources:
        for file in files:
            print(f"Loading {source} {file} data")
            df = spark.read.csv(
                f"s3://spark-databricks-bootcamp/orders/{source}/{file_run}/{source}_{file}.csv",
                header=True,
            )
            df_enriched = (
                df.withColumn("file_date", F.lit(file_run))
                .withColumn("source", F.lit(source))
                .withColumn("ingestion_ts", F.lit(datetime.now()))
                .write.mode("append")
                .saveAsTable(f"retail_data.bronze.{source}_{file}")
            )

In [0]:
def cleanup_old_data():
    for source in sources:
        for file in files:
            print(f"Cleaning up {source} {file} data for {file_run}")
            spark.sql(f"delete from retail_data.bronze.{source}_{file} where file_date = '{file_run}'")

In [0]:
cleanup_old_data()

In [0]:
load_bronze_data()